# Module 06, Unit 02 — Marginals, Conditionals & the Schur Complement

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 06 | Unit 02

| | |
|---|---|
| **Estimated time** | 65–75 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical bridge** | Bayesian updating — Gaussian posterior as a conditional MVN `[BAY]` |
| **Prerequisites** | Module 06 Unit 01; Module 05 Unit 03 (bivariate normal conditionals) |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] Partition a mean vector and covariance matrix into blocks and name each block
- [ ] State the marginal distribution of any sub-vector of an MVN
- [ ] Derive the conditional distribution $\boldsymbol{X}_1 \mid \boldsymbol{X}_2 = \boldsymbol{x}_2$ using the block matrix inverse
- [ ] Identify the Schur complement and explain its role as the conditional covariance
- [ ] Apply the Woodbury identity to update an inverse efficiently after a rank-1 update
- [ ] Derive the Gaussian posterior (Bayesian update) as a conditional MVN

---

## Why This Unit

The MVN's most powerful property is its closure under marginalization and conditioning: the marginal of an MVN is MVN, and the conditional of an MVN given another component is MVN. These two facts are the foundation of Kalman filtering, Gaussian process regression, Bayesian linear regression, and any probabilistic model built on Gaussian priors.

Deriving these distributions requires partitioning the covariance matrix into blocks and inverting the result — an algebraic operation called the **Schur complement**. It is the hardest calculation in this module, and also the most rewarding.

---

## 1. Block Partitioning

Partition the random vector into two sub-vectors:

$$
\boldsymbol{X} = \begin{pmatrix} \boldsymbol{X}_1 \\ \boldsymbol{X}_2 \end{pmatrix}, \qquad \boldsymbol{X}_1 \in \mathbb{R}^{n_1}, \quad \boldsymbol{X}_2 \in \mathbb{R}^{n_2}, \quad n_1 + n_2 = n
$$

Partition the mean and covariance conformably:

$$
\boldsymbol{\mu} = \begin{pmatrix} \boldsymbol{\mu}_1 \\ \boldsymbol{\mu}_2 \end{pmatrix}, \qquad \boldsymbol{\Sigma} = \begin{pmatrix} \boldsymbol{\Sigma}_{11} & \boldsymbol{\Sigma}_{12} \\ \boldsymbol{\Sigma}_{21} & \boldsymbol{\Sigma}_{22} \end{pmatrix}
$$

where:
- $\boldsymbol{\Sigma}_{11} = \text{Cov}(\boldsymbol{X}_1, \boldsymbol{X}_1) \in \mathbb{R}^{n_1 \times n_1}$ — variance of $\boldsymbol{X}_1$
- $\boldsymbol{\Sigma}_{22} = \text{Cov}(\boldsymbol{X}_2, \boldsymbol{X}_2) \in \mathbb{R}^{n_2 \times n_2}$ — variance of $\boldsymbol{X}_2$
- $\boldsymbol{\Sigma}_{12} = \text{Cov}(\boldsymbol{X}_1, \boldsymbol{X}_2) \in \mathbb{R}^{n_1 \times n_2}$ — cross-covariance
- $\boldsymbol{\Sigma}_{21} = \boldsymbol{\Sigma}_{12}^{\top}$ — symmetry of covariance

---

## 2. Marginal Distributions

**Theorem.** If $\boldsymbol{X} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ with the partitioning above, then:

$$
\boldsymbol{X}_1 \sim \mathcal{N}(\boldsymbol{\mu}_1, \boldsymbol{\Sigma}_{11})
$$

$$
\boldsymbol{X}_2 \sim \mathcal{N}(\boldsymbol{\mu}_2, \boldsymbol{\Sigma}_{22})
$$

**Proof sketch.** $\boldsymbol{X}_1 = \boldsymbol{A}\boldsymbol{X}$ where $\boldsymbol{A} = [\boldsymbol{I}_{n_1} \mid \boldsymbol{0}]$ is a selection matrix. Since affine transformations of MVNs are MVN:

$$
\boldsymbol{X}_1 = \boldsymbol{A}\boldsymbol{X} \sim \mathcal{N}(\boldsymbol{A}\boldsymbol{\mu},\ \boldsymbol{A}\boldsymbol{\Sigma}\boldsymbol{A}^{\top}) = \mathcal{N}(\boldsymbol{\mu}_1,\ \boldsymbol{\Sigma}_{11}) \qquad \square
$$

**Key consequence:** The marginal of an MVN only requires the corresponding block of the covariance matrix. All off-diagonal blocks $\boldsymbol{\Sigma}_{12}$ are irrelevant for the marginals — but they are essential for the conditionals.

---

## 3. Conditional Distributions and the Schur Complement

### 3.1 The Result

**Theorem.** Under the same partitioning:

$$
\boldsymbol{X}_1 \mid \boldsymbol{X}_2 = \boldsymbol{x}_2 \sim \mathcal{N}\!\left(\boldsymbol{\mu}_{1|2},\ \boldsymbol{\Sigma}_{1|2}\right)
$$

where:

$$
\boldsymbol{\mu}_{1|2} = \boldsymbol{\mu}_1 + \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}(\boldsymbol{x}_2 - \boldsymbol{\mu}_2)
$$

$$
\boldsymbol{\Sigma}_{1|2} = \boldsymbol{\Sigma}_{11} - \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}\boldsymbol{\Sigma}_{21}
$$

The matrix $\boldsymbol{\Sigma}_{11} - \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}\boldsymbol{\Sigma}_{21}$ is the **Schur complement** of $\boldsymbol{\Sigma}_{22}$ in $\boldsymbol{\Sigma}$.

### 3.2 Reading the Formulas

**Conditional mean** $\boldsymbol{\mu}_{1|2} = \boldsymbol{\mu}_1 + \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}(\boldsymbol{x}_2 - \boldsymbol{\mu}_2)$:

- Starts from the prior mean $\boldsymbol{\mu}_1$
- Corrects by the **regression coefficient** matrix $\boldsymbol{B} = \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}$
- Applied to the **deviation** $(\boldsymbol{x}_2 - \boldsymbol{\mu}_2)$ of the observed value from its mean
- This is exactly multivariate linear regression of $\boldsymbol{X}_1$ on $\boldsymbol{X}_2$

**Conditional covariance** $\boldsymbol{\Sigma}_{1|2} = \boldsymbol{\Sigma}_{11} - \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}\boldsymbol{\Sigma}_{21}$:

- Starts from the marginal variance $\boldsymbol{\Sigma}_{11}$
- **Subtracts** the variance explained by knowing $\boldsymbol{X}_2$
- Does **not** depend on the observed value $\boldsymbol{x}_2$ — the conditional variance is fixed regardless of what $\boldsymbol{X}_2$ equals (homoskedasticity of the MVN)
- $\boldsymbol{\Sigma}_{1|2} \leq \boldsymbol{\Sigma}_{11}$ in the PSD sense — conditioning never increases variance

### 3.3 Derivation via Block Matrix Inversion

The conditional density $f_{\boldsymbol{X}_1|\boldsymbol{X}_2}(\boldsymbol{x}_1|\boldsymbol{x}_2) = f(\boldsymbol{x}_1, \boldsymbol{x}_2)/f_{\boldsymbol{X}_2}(\boldsymbol{x}_2)$. To evaluate this, we need $\boldsymbol{\Sigma}^{-1}$ in block form. The key formula is the **block matrix inverse**:

$$
\begin{pmatrix} \boldsymbol{\Sigma}_{11} & \boldsymbol{\Sigma}_{12} \\ \boldsymbol{\Sigma}_{21} & \boldsymbol{\Sigma}_{22} \end{pmatrix}^{-1}
= \begin{pmatrix} \boldsymbol{\Sigma}_{1|2}^{-1} & -\boldsymbol{\Sigma}_{1|2}^{-1}\boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1} \\ -\boldsymbol{\Sigma}_{22}^{-1}\boldsymbol{\Sigma}_{21}\boldsymbol{\Sigma}_{1|2}^{-1} & \boldsymbol{\Sigma}_{22}^{-1} + \boldsymbol{\Sigma}_{22}^{-1}\boldsymbol{\Sigma}_{21}\boldsymbol{\Sigma}_{1|2}^{-1}\boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1} \end{pmatrix}
$$

where $\boldsymbol{\Sigma}_{1|2} = \boldsymbol{\Sigma}_{11} - \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}\boldsymbol{\Sigma}_{21}$ is the Schur complement.

Substituting into the joint density and completing the square in $\boldsymbol{x}_1$ yields the conditional density as a Gaussian with mean $\boldsymbol{\mu}_{1|2}$ and covariance $\boldsymbol{\Sigma}_{1|2}$.

### 3.4 Reduction to the Bivariate Case

For $n_1 = n_2 = 1$ with $\boldsymbol{\Sigma}_{11} = \sigma_X^2$, $\boldsymbol{\Sigma}_{22} = \sigma_Y^2$, $\boldsymbol{\Sigma}_{12} = \rho\sigma_X\sigma_Y$:

$$
\boldsymbol{\Sigma}_{1|2} = \sigma_X^2 - \frac{(\rho\sigma_X\sigma_Y)^2}{\sigma_Y^2} = \sigma_X^2(1-\rho^2)
$$

$$
\boldsymbol{\mu}_{1|2} = \mu_X + \frac{\rho\sigma_X\sigma_Y}{\sigma_Y^2}(y - \mu_Y) = \mu_X + \rho\frac{\sigma_X}{\sigma_Y}(y - \mu_Y)
$$

This matches the bivariate conditional from Module 05 Unit 03 exactly. ✓

---

## 4. The Woodbury Identity

In Bayesian inference and Kalman filtering, we frequently need to update $\boldsymbol{\Sigma}^{-1}$ after receiving new information — a low-rank update to the covariance. The **Woodbury matrix identity** does this efficiently:

$$
(\boldsymbol{A} + \boldsymbol{U}\boldsymbol{C}\boldsymbol{V})^{-1} = \boldsymbol{A}^{-1} - \boldsymbol{A}^{-1}\boldsymbol{U}(\boldsymbol{C}^{-1} + \boldsymbol{V}\boldsymbol{A}^{-1}\boldsymbol{U})^{-1}\boldsymbol{V}\boldsymbol{A}^{-1}
$$

**Why this matters:** If $\boldsymbol{A}$ is $n \times n$ and $\boldsymbol{U}, \boldsymbol{V}$ are $n \times k$ with $k \ll n$, inverting $\boldsymbol{A} + \boldsymbol{U}\boldsymbol{C}\boldsymbol{V}$ directly costs $O(n^3)$. The Woodbury formula reduces this to one $k \times k$ inversion $O(k^3)$ plus matrix multiplications — a massive saving when $k$ is small. In Gaussian process regression with $n$ observations, the Woodbury identity is what makes the predictive update tractable.

---

## 5. Statistical Bridge — Bayesian Updating as a Conditional MVN `[BAY]`

### 5.1 The Setup

Bayesian linear regression with a Gaussian prior. Let:

$$
\boldsymbol{\theta} \sim \mathcal{N}(\boldsymbol{m}_0, \boldsymbol{S}_0) \qquad \text{(prior on parameters)}
$$
$$
\boldsymbol{y} \mid \boldsymbol{\theta} \sim \mathcal{N}(\boldsymbol{X}\boldsymbol{\theta},\ \sigma^2\boldsymbol{I}) \qquad \text{(Gaussian likelihood)}
$$

We want $p(\boldsymbol{\theta} \mid \boldsymbol{y})$ — the posterior.

### 5.2 Joint Distribution of $(\boldsymbol{\theta}, \boldsymbol{y})$

The joint $(\boldsymbol{\theta}, \boldsymbol{y})$ is also Gaussian:

$$
\begin{pmatrix} \boldsymbol{\theta} \\ \boldsymbol{y} \end{pmatrix} \sim \mathcal{N}\!\left(\begin{pmatrix}\boldsymbol{m}_0 \\ \boldsymbol{X}\boldsymbol{m}_0\end{pmatrix},\ \begin{pmatrix}\boldsymbol{S}_0 & \boldsymbol{S}_0\boldsymbol{X}^{\top} \\ \boldsymbol{X}\boldsymbol{S}_0 & \boldsymbol{X}\boldsymbol{S}_0\boldsymbol{X}^{\top} + \sigma^2\boldsymbol{I}\end{pmatrix}\right)
$$

Identifying the blocks: $\boldsymbol{\Sigma}_{11} = \boldsymbol{S}_0$, $\boldsymbol{\Sigma}_{12} = \boldsymbol{S}_0\boldsymbol{X}^{\top}$, $\boldsymbol{\Sigma}_{22} = \boldsymbol{X}\boldsymbol{S}_0\boldsymbol{X}^{\top} + \sigma^2\boldsymbol{I}$.

### 5.3 The Posterior

Applying the conditional MVN formula:

$$
\boldsymbol{\theta} \mid \boldsymbol{y} \sim \mathcal{N}(\boldsymbol{m}_n, \boldsymbol{S}_n)
$$

where:

$$
\boldsymbol{S}_n = \left(\boldsymbol{S}_0^{-1} + \frac{1}{\sigma^2}\boldsymbol{X}^{\top}\boldsymbol{X}\right)^{-1}
$$

$$
\boldsymbol{m}_n = \boldsymbol{S}_n\left(\boldsymbol{S}_0^{-1}\boldsymbol{m}_0 + \frac{1}{\sigma^2}\boldsymbol{X}^{\top}\boldsymbol{y}\right)
$$

**Reading the posterior:**
- $\boldsymbol{S}_n^{-1} = \boldsymbol{S}_0^{-1} + \frac{1}{\sigma^2}\boldsymbol{X}^{\top}\boldsymbol{X}$: the posterior precision is the prior precision plus the data precision. Precision **adds** when combining independent information.
- $\boldsymbol{m}_n$: the posterior mean is a precision-weighted average of the prior mean $\boldsymbol{m}_0$ and the data-driven MLE direction $\frac{1}{\sigma^2}\boldsymbol{X}^{\top}\boldsymbol{y}$.
- **Ridge connection:** When $\boldsymbol{S}_0 = \frac{\sigma^2}{\lambda}\boldsymbol{I}$, the posterior mean becomes $\boldsymbol{m}_n = (\boldsymbol{X}^{\top}\boldsymbol{X} + \lambda\boldsymbol{I})^{-1}\boldsymbol{X}^{\top}\boldsymbol{y} = \hat{\boldsymbol{\beta}}_{\text{Ridge}}$. **Ridge regression is Bayesian linear regression with a spherical Gaussian prior.** The regularization parameter $\lambda$ controls the prior precision.

> **This is the deepest thread connection in the course.** The Ridge solution from Module 04 Unit 02, the gradient of the log-likelihood from Module 03 Unit 01, and the Bayesian posterior from this unit are three descriptions of the same object — they are related by the conditional MVN formula applied to a Gaussian prior-likelihood model.

---

## Python: Visualization & Verification

The cells below demonstrate and visualize the concepts above. **Run them in order.**

To run this notebook interactively, click the **rocket icon** at the top of the page and select **Open in Colab**.

In [ ]:
import subprocess, sys
for pkg in ["numpy", "matplotlib", "sympy", "scipy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy import Matrix, symbols, sqrt, simplify
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (9,6), 'font.size': 11,
                     'axes.titlesize': 13, 'lines.linewidth': 2, 'figure.dpi': 120})
TS_BLUE='#1e64b4'; TS_AMBER='#c87814'; TS_GREEN='#1e8c50'
TS_GRAY='#64646e'; TS_RED='#b41e1e'; TS_LIGHT='#f5f7fa'
print('Setup complete.')

### Section 1 — Symbolic Schur Complement Verification

Verify the block matrix inverse formula and confirm the conditional distribution formulas match the bivariate normal from Module 05 Unit 03.

In [ ]:
# ============================================================
# Section 1 — Symbolic Schur complement and block inverse
# ============================================================
# Use 2x2 block structure with scalar blocks for clarity
s11, s12, s22 = sp.symbols('Sigma_11 Sigma_12 Sigma_22', positive=True)
rho_s = sp.Symbol('rho', positive=True)

# Bivariate case: Sigma_11=σ_x², Sigma_12=ρσ_xσ_y, Sigma_22=σ_y²
sx2, sy2, rho_v = sp.symbols('sigma_x^2 sigma_y^2 rho', positive=True)
Sigma = sp.Matrix([[sx2, rho_v*sp.sqrt(sx2*sy2)],
                   [rho_v*sp.sqrt(sx2*sy2), sy2]])

# Schur complement Σ_{1|2} = Σ_{11} - Σ_{12}Σ_{22}⁻¹Σ_{21}
S11 = Sigma[0,0]
S12 = Sigma[0,1]
S21 = Sigma[1,0]
S22 = Sigma[1,1]

Schur = S11 - S12 * (1/S22) * S21
Schur_simplified = sp.simplify(Schur)
print('Schur complement Σ_{1|2} = Σ_{11} − Σ_{12}Σ_{22}⁻¹Σ_{21}:')
sp.pprint(Schur_simplified)
print('= σ_x²(1−ρ²)?', sp.simplify(Schur_simplified - sx2*(1-rho_v**2)) == 0)

# Verify block matrix inverse formula: Sigma @ Sigma_inv = I
print('\nVerifying block inverse formula...')
Sigma_inv_formula = Sigma.inv()
product = sp.simplify(Sigma * Sigma_inv_formula)
print('Σ · Σ⁻¹ = I?', product == sp.eye(2))

# Conditional regression coefficient B = Σ₁₂ Σ₂₂⁻¹
B = S12 / S22
B_simplified = sp.simplify(B)
print('\nRegression coefficient B = Σ₁₂Σ₂₂⁻¹:')
sp.pprint(B_simplified)
print('= ρ·σ_x/σ_y?', sp.simplify(B_simplified - rho_v*sp.sqrt(sx2/sy2)) == 0)

**What do you see?** The Schur complement for the bivariate case equals $\sigma_X^2(1-\rho^2)$ — exactly the conditional variance from Module 05 Unit 03. The regression coefficient $B = \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1} = \rho\sigma_X/\sigma_Y$ — which is the slope of the linear regression of $X_1$ on $X_2$, scaled by standard deviations.

### Section 2 — Marginal and Conditional Distributions: Numerical Verification

For a 4-dimensional MVN, verify the marginal and conditional distribution formulas numerically using Monte Carlo sampling, and visualize how conditioning sharpens the distribution.

In [ ]:
# ============================================================
# Section 2 — Marginal and conditional: numerical verification
# ============================================================
rng = np.random.default_rng(42)
N = 100_000

# 4D MVN: X = (X1, X2, X3, X4)
# Partition: X_A = (X1, X2), X_B = (X3, X4)
mu_full = np.array([1.0, -0.5, 2.0, 0.5])
Sigma_full = np.array([
    [2.0,  1.2,  0.8, -0.3],
    [1.2,  1.5,  0.5,  0.2],
    [0.8,  0.5,  3.0,  1.0],
    [-0.3, 0.2,  1.0,  1.2],
])

# Block partitioning
mu1 = mu_full[:2];    mu2 = mu_full[2:]
S11 = Sigma_full[:2,:2]; S12 = Sigma_full[:2,2:]
S21 = Sigma_full[2:,:2]; S22 = Sigma_full[2:,2:]

# Analytical marginal of X_A
mu_A_anal   = mu1
Sigma_A_anal = S11

# Analytical conditional X_A | X_B = x_b
x_b_obs = np.array([1.5, 0.0])   # observed X_B value
S22_inv  = np.linalg.inv(S22)
B_reg    = S12 @ S22_inv           # regression coefficient (2×2)
Schur    = S11 - S12 @ S22_inv @ S21  # conditional covariance
mu_cond  = mu1 + B_reg @ (x_b_obs - mu2)

# Monte Carlo samples
samples = rng.multivariate_normal(mu_full, Sigma_full, N)
X_A = samples[:, :2]
X_B = samples[:, 2:]

# Empirical marginal stats for X_A
mu_A_mc    = X_A.mean(axis=0)
Sigma_A_mc = np.cov(X_A.T)

# Conditional samples: filter by X_B ≈ x_b_obs
tol = 0.25
mask = np.all(np.abs(X_B - x_b_obs) < tol, axis=1)
X_A_cond = X_A[mask]

print('=== Marginal of X_A = (X1, X2) ===')
print(f'Analytical μ_A = {mu_A_anal}')
print(f'MC         μ_A = {np.round(mu_A_mc, 4)}')
print(f'Analytical Σ_A =\n{Sigma_A_anal}')
print(f'MC         Σ_A =\n{np.round(Sigma_A_mc, 4)}')

print(f'\n=== Conditional X_A | X_B = {x_b_obs} ===')
print(f'Analytical μ_{{1|2}} = {np.round(mu_cond, 4)}')
print(f'MC         μ_{{1|2}} = {np.round(X_A_cond.mean(axis=0), 4)}  (from {mask.sum()} samples)')
print(f'Analytical Σ_{{1|2}} =\n{np.round(Schur, 4)}')
print(f'MC         Σ_{{1|2}} =\n{np.round(np.cov(X_A_cond.T), 4)}')

# Visualization: marginal vs conditional in first variable
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, j, varname in [(axes[0], 0, '$X_1$'), (axes[1], 1, '$X_2$')]:
    xp = np.linspace(mu_full[j]-4, mu_full[j]+4, 400)
    # Marginal
    f_marg = stats.norm.pdf(xp, mu_A_anal[j], np.sqrt(S11[j,j]))
    ax.plot(xp, f_marg, color=TS_BLUE, lw=2.5, label=f'Marginal {varname}')
    ax.hist(X_A[:,j], bins=60, density=True, alpha=0.25,
            color=TS_BLUE, edgecolor='white')
    # Conditional
    f_cond = stats.norm.pdf(xp, mu_cond[j], np.sqrt(Schur[j,j]))
    ax.plot(xp, f_cond, color=TS_RED, lw=2.5, ls='--',
            label=f'Conditional $X_{j+1}|X_B={x_b_obs}$')
    if len(X_A_cond) > 10:
        ax.hist(X_A_cond[:,j], bins=20, density=True, alpha=0.35,
                color=TS_RED, edgecolor='white')
    ax.set_xlabel(varname); ax.set_ylabel('density')
    ax.set_title(f'{varname}: marginal vs conditional')
    ax.legend(fontsize=9)

plt.suptitle('Conditioning sharpens the distribution — '
             r'$\Sigma_{1|2} \leq \Sigma_{11}$', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?** The analytical marginal and conditional moments match the Monte Carlo estimates closely. The conditional distribution (red dashed) is narrower than the marginal (blue) — conditioning on $\boldsymbol{X}_B$ reduces uncertainty about $\boldsymbol{X}_A$. The conditional mean has shifted from the marginal mean because $\boldsymbol{X}_A$ and $\boldsymbol{X}_B$ are correlated — the observed $\boldsymbol{x}_B$ updates our belief about $\boldsymbol{X}_A$ through the regression coefficient $\boldsymbol{B} = \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}$.

### Section 3 — Bayesian Linear Regression: Prior, Likelihood, and Posterior

Implement the Gaussian posterior update formula from Section 5 and visualize how the posterior evolves as data arrives sequentially — the Kalman-style recursive update.

In [ ]:
# ============================================================
# Section 3 — Bayesian linear regression: sequential updating
# ============================================================
rng = np.random.default_rng(7)

# True parameters and noise
theta_true = np.array([2.0, -1.5])   # intercept, slope
sigma2_noise = 0.5

# Prior: θ ~ N(m0, S0)
m0 = np.zeros(2)
S0 = 4.0 * np.eye(2)   # broad prior

# Generate data sequentially
n_total = 50
x_data = rng.uniform(-3, 3, n_total)
X_design = np.column_stack([np.ones(n_total), x_data])   # (n, 2)
y_data = X_design @ theta_true + rng.normal(0, np.sqrt(sigma2_noise), n_total)

def gaussian_posterior(X, y, m0, S0, sigma2):
    """Compute Gaussian posterior: θ|y ~ N(m_n, S_n)."""
    S0_inv = np.linalg.inv(S0)
    S_n_inv = S0_inv + X.T @ X / sigma2
    S_n = np.linalg.inv(S_n_inv)
    m_n = S_n @ (S0_inv @ m0 + X.T @ y / sigma2)
    return m_n, S_n

# Compute posterior at several data sizes
ns_plot = [1, 5, 15, 50]
posteriors = {}
for n in ns_plot:
    m_n, S_n = gaussian_posterior(
        X_design[:n], y_data[:n], m0, S0, sigma2_noise)
    posteriors[n] = (m_n, S_n)

# --- Plot posterior distributions in parameter space ---
theta0_grid = np.linspace(-1, 5, 200)
theta1_grid = np.linspace(-4, 1, 200)
T0, T1 = np.meshgrid(theta0_grid, theta1_grid)
pts = np.stack([T0.ravel(), T1.ravel()], axis=1)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

for ax, n in zip(axes.flat, ns_plot):
    m_n, S_n = posteriors[n]
    rv = stats.multivariate_normal(m_n, S_n)
    Z = rv.pdf(pts).reshape(200, 200)

    ax.contourf(T0, T1, Z, levels=12, cmap='Blues', alpha=0.70)
    ax.contour(T0, T1, Z, levels=12, colors=[TS_GRAY], alpha=0.4, linewidths=0.7)

    # True parameter
    ax.plot(*theta_true, '*', color=TS_RED, markersize=14, zorder=6,
            label=fr'True $\theta = {theta_true}$')
    # Posterior mean
    ax.plot(*m_n, 'o', color=TS_AMBER, markersize=10, zorder=6,
            label=fr'$\hat{{m}}_n = ({m_n[0]:.2f}, {m_n[1]:.2f})$')

    std_diag = np.sqrt(np.diag(S_n))
    ax.set_title(f'n = {n} observations\n'
                 fr'$\pm 1\sigma$: $\theta_0 \in [{m_n[0]-std_diag[0]:.2f}, {m_n[0]+std_diag[0]:.2f}]$, '
                 fr'$\theta_1 \in [{m_n[1]-std_diag[1]:.2f}, {m_n[1]+std_diag[1]:.2f}]$',
                 fontsize=9)
    ax.set_xlabel(r'$\theta_0$ (intercept)')
    ax.set_ylabel(r'$\theta_1$ (slope)')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_xlim(-1, 5); ax.set_ylim(-4, 1)

plt.suptitle('Gaussian posterior $\\theta | y \\sim \\mathcal{N}(m_n, S_n)$'
             ' — concentrating around the truth with more data',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

# Ridge connection
lam = sigma2_noise / S0[0,0]   # λ = σ²/s0²
beta_ridge = np.linalg.solve(X_design.T @ X_design + lam*np.eye(2),
                              X_design.T @ y_data)
m_n_full, _ = gaussian_posterior(X_design, y_data, m0, S0, sigma2_noise)
print(f'Ridge β̂ (λ={lam:.2f})  = {np.round(beta_ridge, 5)}')
print(f'Bayesian posterior m_n = {np.round(m_n_full, 5)}')
print(f'Identical?  {np.allclose(beta_ridge, m_n_full)}')

**What do you see?**

- With $n=1$ observation the posterior is nearly identical to the prior — one data point barely shifts the broad Gaussian distribution over $\boldsymbol{\theta}$.
- As $n$ grows, the posterior concentrates around the true parameter vector (red star). The uncertainty ellipse shrinks — $\boldsymbol{S}_n \to \boldsymbol{0}$ as $n \to \infty$.
- The printed confirmation at the bottom: with a spherical Gaussian prior, the Bayesian posterior mean **exactly equals** the Ridge regression estimate. Ridge regression is not an ad hoc regularization device — it is Bayesian linear regression under a Gaussian prior, and the regularization parameter $\lambda$ encodes the prior precision $\sigma^2/s_0^2$.

In [ ]:
# Extension workspace
# Suggestions:
# 1. Verify the Woodbury identity numerically: for n=100, p=2,
#    compare (XᵀX + λI)⁻¹ computed directly vs via Woodbury
#    applied to (λI + XᵀX)⁻¹. Time both for large n.
#
# 2. Kalman filter: implement one step of the Kalman update
#    as a conditional MVN. State: X_t; observation: Y_t = H X_t + noise.
#    Compute the posterior X_t|Y_t using the block conditional formula.
#
# 3. Sequential Bayesian update: instead of batching all n observations,
#    update (m, S) one observation at a time using the rank-1 Woodbury update.
#    Verify you reach the same posterior as the batch formula.


---

## Summary

| Concept | Statement |
|---|---|
| Marginal MVN | $\boldsymbol{X}_1 \sim \mathcal{N}(\boldsymbol{\mu}_1, \boldsymbol{\Sigma}_{11})$ — just take the sub-block |
| Conditional mean | $\boldsymbol{\mu}_{1\|2} = \boldsymbol{\mu}_1 + \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}(\boldsymbol{x}_2 - \boldsymbol{\mu}_2)$ |
| Schur complement | $\boldsymbol{\Sigma}_{1\|2} = \boldsymbol{\Sigma}_{11} - \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}\boldsymbol{\Sigma}_{21}$ |
| Conditioning never increases variance | $\boldsymbol{\Sigma}_{1\|2} \leq \boldsymbol{\Sigma}_{11}$ (PSD sense) |
| Regression coefficient | $\boldsymbol{B} = \boldsymbol{\Sigma}_{12}\boldsymbol{\Sigma}_{22}^{-1}$ — slope of $\boldsymbol{X}_1$ on $\boldsymbol{X}_2$ |
| Gaussian posterior precision | $\boldsymbol{S}_n^{-1} = \boldsymbol{S}_0^{-1} + \frac{1}{\sigma^2}\boldsymbol{X}^{\top}\boldsymbol{X}$ |
| Ridge = Gaussian posterior | $\hat{\boldsymbol{\beta}}_{\text{Ridge}} = \boldsymbol{m}_n$ when prior is spherical Gaussian |

**Up next:** Unit 03 — MLE for the MVN: Putting It All Together

We differentiate the MVN log-likelihood with respect to both $\boldsymbol{\mu}$ and $\boldsymbol{\Sigma}$, derive the MLE estimators from first principles, and confirm that the Hessian is negative definite — confirming every critical point is a maximum.

---
*Module 06, Unit 02 — Threat Surfaces | fischer³ Education*